In [1]:
import os
import sys
import ctypes
import resource

# =============================================================================
# 0. THE CURE: PRE-IMPORT CUDA ENVIRONMENT HACK
# =============================================================================
# Absolute paths from your terminal search
cuda_root = "/oscar/rt/9.6/25/spack/x86_64_v3/cuda-12.9.0-cinrl2oeqemd3szbcakkugp2vtk2fh5t"
nvvm_lib_dir = os.path.join(cuda_root, "nvvm", "lib64")
nvrtc_lib_dir = os.path.join(cuda_root, "targets", "x86_64-linux", "lib")
standard_lib_dir = os.path.join(cuda_root, "lib64")

# Force these into the environment BEFORE importing numba/ipie/tf
os.environ['CUDA_HOME'] = cuda_root
os.environ['CPATH'] = os.path.join(cuda_root, 'include')
os.environ['PATH'] = os.path.join(cuda_root, 'bin') + ":" + os.environ.get('PATH', '')

os.environ['NUMBA_CUDA_DRIVER'] = "/lib64/libcuda.so"

# Combine Spack paths with the system /lib64 path
os.environ['LD_LIBRARY_PATH'] = f"{nvvm_lib_dir}:{nvrtc_lib_dir}:{standard_lib_dir}:/lib64:" + os.environ.get('LD_LIBRARY_PATH', '')

# Force IPIE to look for GPU kernels
os.environ["IPIE_USE_GPU"] = "1"

# THE "PROCESS-HACK": Manually load the library into global process memory
try:
    ctypes.CDLL(os.path.join(nvvm_lib_dir, "libnvvm.so"), mode=ctypes.RTLD_GLOBAL)
    print("libnvvm.so successfully injected into process memory.")
except Exception as e:
    print(f"Manual injection failed: {e}")

# =============================================================================
# 1. IMPORTS
# =============================================================================
# NOW (and only now) import the heavy hitters
import time
import json
import numpy as np

# PySCF & IPIE Imports
from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler

# IPIE Analysis Imports
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable

# Check for CuPy
try:
    import cupy as cp
    has_cupy = True
except ImportError:
    cp = None
    has_cupy = False

# =============================================================================
# 2. CONFIGURATION
# =============================================================================
N_ATOMS = 70
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
TEST_SEED = 999
NUM_BLOCKS = 50
STEPS_PER_BLOCK = 1

# =============================================================================
# 3. SETUP MPI & GPU MAPPING
# =============================================================================
comm = MPIHandler()
rank = comm.rank

# [CRITICAL] Map MPI ranks to GPUs on the node
if has_cupy:
    num_gpus = cp.cuda.runtime.getDeviceCount()
    if num_gpus > 0:
        device_id = rank % num_gpus
        cp.cuda.Device(device_id).use()
        if rank == 0:
            print(f">>> Found {num_gpus} GPUs. Mapping Rank {rank} to Device {device_id}")

chk_file = f"scf_h{N_ATOMS}.chk"
ham_file = f"ham_h{N_ATOMS}.h5"
wfn_file = f"wfn_h{N_ATOMS}.h5"

if rank == 0:
    print(f">>> STARTING GPU CLASSICAL AFQMC BASELINE FOR: {SYSTEM_NAME}")
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.chkfile = chk_file
    
    if not os.path.exists(chk_file) or not os.path.exists(wfn_file):
        print(f"[Main] Running PySCF and generating integrals...")
        CURRENT_HF_ENERGY = mf.kernel()
        # Explicitly setting chol_cut to 1e-5 to match perfectly
        gen_ipie_input_from_pyscf_chk(chk_file, hamil_file=ham_file, wfn_file=wfn_file, verbose=0, chol_cut=1e-5)
    else:
        CURRENT_HF_ENERGY = lib.chkfile.load(chk_file, 'scf/e_tot')
    print(f"[Main] Current PySCF HF Energy: {CURRENT_HF_ENERGY:.6f} Ha")
else:
    CURRENT_HF_ENERGY = None

comm.comm.Barrier()

# =============================================================================
# 4. INITIALIZE AFQMC (GPU ENABLED)
# =============================================================================
afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2),
    ham_file=ham_file,
    wfn_file=wfn_file,
    num_blocks=NUM_BLOCKS,
    num_steps_per_block=STEPS_PER_BLOCK,
    num_walkers=WALKERS,
    seed=TEST_SEED,
    verbose=0
)

# Force AFQMC to use GPU backend
if has_cupy:
    afqmc.cuda = True

afqmc.mpi_handler = comm
if comm.size > 1:
    local_walkers = WALKERS // comm.size
    if rank < (WALKERS % comm.size): local_walkers += 1
    afqmc.nwalkers = local_walkers

# =============================================================================
# 5. RUN WITH PROFILING
# =============================================================================
if rank == 0:
    print("\n" + "#"*60 + "\n### STARTING CLASSICAL IPIE GPU PRODUCTION RUN ###\n" + "#"*60)
    start_time = time.perf_counter()

# This will now use CuPy kernels if afqmc.cuda is True
afqmc.run()

if rank == 0:
    end_time = time.perf_counter()
    
    # 1. True Peak Host RAM (in MB)
    peak_host_ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    
    # 2. True Peak GPU VRAM (CuPy only, since there's no TF here)
    cp_vram_mb = 0
    if has_cupy:
        try:
            cp_vram_mb = cp.get_default_memory_pool().total_bytes() / (1024 ** 2)
        except AttributeError: pass
    
    total_peak_vram_mb = cp_vram_mb

# =============================================================================
# 6. DATA EXTRACTION, ANALYSIS & SAVING
# =============================================================================
if rank == 0:
    print("\n" + "="*50)
    print(">>> PERFORMING AUTOCORRELATION ANALYSIS")
    print("="*50)
    
    final_energy = None
    final_error = None
    
    try:
        estimator_filename = afqmc.estimators.filename
        qmc_data = extract_observable(estimator_filename, "energy")
        y_energy = qmc_data["ETotal"][1:]
        df_ac = reblock_by_autocorr(y_energy, verbose=0)
        
        final_energy = float(df_ac["ETotal_ac"].iloc[0])
        final_error = float(df_ac["ETotal_error_ac"].iloc[0])
        
        print(f"  Final Reblocked Energy: {final_energy:.6f} Ha")
        print(f"  Final Reblocked Error:  {final_error:.6f} Ha")
        
    except Exception as e:
        print(f"  [Warning] Autocorrelation analysis failed: {e}")
        print("  Falling back to raw standard error of the mean.")
        if 'y_energy' in locals() and len(y_energy) > 0:
            final_energy = float(np.mean(y_energy))
            final_error = float(np.std(y_energy) / np.sqrt(len(y_energy)))
            print(f"  Fallback Raw Mean Energy: {final_energy:.6f} ± {final_error:.6f} Ha")

    # -----------------------------
    # TIMING & FLOPs CALCULATION
    # -----------------------------
    total_time = end_time - start_time
    time_per_block = total_time / NUM_BLOCKS

    # Physics Overhead
    flops_physics_per_walker = 8 * (N_ATOMS ** 3) 
    
    # Total Hardware Operations (Safely using global WALKERS and STEPS_PER_BLOCK)
    flops_total_per_step = flops_physics_per_walker * WALKERS
    total_flops_run = flops_total_per_step * NUM_BLOCKS * STEPS_PER_BLOCK
    teraflops_per_sec = (total_flops_run / total_time) / 1e12

    metrics = {
        "system": SYSTEM_NAME,
        "backend": "GPU (Classical IPIE)",
        "n_atoms": N_ATOMS,
        "total_walkers": WALKERS,
        "num_blocks": NUM_BLOCKS,
        "results": {
            "final_energy_ha": final_energy,
            "final_error_ha": final_error
        },
        "timing": {
            "total_wall_time_sec": round(total_time, 4),
            "time_per_block_sec": round(time_per_block, 4)
        },
        "memory": {
            "peak_host_ram_mb": round(peak_host_ram_mb, 2),
            "peak_gpu_vram_mb": round(total_peak_vram_mb, 2),
            "gpu_vram_split": {
                "cupy_mb": round(cp_vram_mb, 2)
            }
        },
        "compute": {
            "flops_physics_per_walker": flops_physics_per_walker,
            "total_tflops_executed": round(total_flops_run / 1e12, 6),
            "tflops_throughput": round(teraflops_per_sec, 6)
        }
    }

    print("\n" + "="*50)
    print(">>> BASELINE SCALING METRICS SUMMARY")
    print("="*50)
    print(f"  Final Energy:         {final_energy} ± {final_error} Ha")
    print(f"  Total Wall Time:      {total_time:.4f} sec")
    print(f"  Peak Host RAM:        {peak_host_ram_mb:.2f} MB")
    print(f"  Peak GPU VRAM:        {total_peak_vram_mb:.2f} MB (CuPy: {cp_vram_mb:.1f})")
    print(f"  TFLOP/s Throughput:   {teraflops_per_sec:.6f}")
    print("="*50)

    save_path = f"baseline_metrics_GPU_{SYSTEM_NAME}.json"
    with open(save_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved to {save_path}")

libnvvm.so successfully injected into process memory.
>>> Found 1 GPUs. Mapping Rank 0 to Device 0
>>> STARTING GPU CLASSICAL AFQMC BASELINE FOR: H70
[Main] Current PySCF HF Energy: -34.978273 Ha
# random seed is 999

############################################################
### STARTING CLASSICAL IPIE GPU PRODUCTION RUN ###
############################################################
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -7.1635481297651597e+04  2.0480000000000000e+03 -3.4978262352368944e+01 -2.1845162546255207e+02  1.8347336311018307e+02
                1   2.1435243963992098e+03  2.0480000000000000e+03 -1.8238236819074174e+01 -7.4994348760375433e+04  2.1435243963992098e+03 -3.4986468493829307e+01 -2.1845164555619940e+02 